# IPP Documentation Assessment Data Analysis

Analysis of student documentation submissions and grader feedback, IPP 2013 to 2024. Implemented with a custom assessment `.csv` parser.

In [ ]:
import logging
from pathlib import Path

from IPython.display import display
from scripts import dataset_analysis as da
from scripts import visual_utils as vu
from scripts.assessment_data import dataset_parser

logging.basicConfig(level=logging.INFO, force=True)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

vu.configure_plot_style()

CLEAN_DATA_CSV = Path("../data/clean_ipp_data.csv")

if not CLEAN_DATA_CSV.exists():
    raise FileNotFoundError(f"Missing cleaned dataset CSV: {CLEAN_DATA_CSV}")

df = da.load_clean_data(CLEAN_DATA_CSV)
proj = da.build_project_df(df)

## 1. Dataset Overview

In [ ]:
da.analyse_volume(df, proj)

### Task Variants

The `php`/`py` labels were renamed to `par`/`int` in 2017. In 2017 specifically, int was the one and only Task.

In [ ]:
ax = da.visualise_task_variant_distribution(proj)

### Document Format

PDF was introduced in 2018, before that all submissions were Markdown.

In [ ]:
ax = da.visualise_doc_type_distribution(proj)

## 2. Score Outcomes

### Documentation Deduction Distribution

Most students receive zero or near-zero deductions.

In [ ]:
ax = da.visualise_total_impact_distribution(proj)
da.summarise_score_imbalance(proj)

### Score by Year and Task Variant

Mean documentation score (% of maximum) per year. Shaded area shows 95% confidence interval.

In [ ]:
ax = da.visualise_score_trend_by_year_variant(proj)

### Score Distribution by Task Variant
Whiskers extend to the 5th/95th percentile.

In [ ]:
ax = da.visualise_score_distribution_by_variant(proj)

Bonuses can bring documentation score over 100%

### Format Impact

PDF students score higher on documentation, the cumulative distribution sits mostly above Markdown.

In [ ]:
format_impact_df = da.analyse_format_impact(proj)
ax = da.visualise_format_impact(format_impact_df)
da.summarise_format_impact(format_impact_df)

## 3. Code Frequency

How often each grading code appears across all submissions.

In [ ]:
ax = da.visualise_code_frequency(df, n_codes=20)

## 4. Code Impact

Impact values are expressed as a fraction of the year/variant maximum, making them comparable across years. Shared penalties (e.g. `STRUCT/SHORT`) are excluded.

### Most and Least Impactful Codes

In [ ]:
impact_stats = da.analyse_impact_statistics(df)
frequent = impact_stats[impact_stats["count"] >= 10]

print("Top 10 harshest codes:")
display(frequent.head(10))

print("\nCodes almost always applied as actual deductions:")
display(
    frequent[
        (frequent["pct_no_impact"] < 0.15) & (frequent["median"] < 0)
    ].sort_values("count", ascending=False).head(5)
)

print("\nCodes almost always applied as bonus points:")
display(
    frequent[frequent["median"] > 0].sort_values("count", ascending=False).head(5)
)

print("\nCodes almost always applied as informative warnings:")
display(
    impact_stats[
        ((impact_stats["pct_no_impact"] >= 0.80) | (impact_stats["median"] == 0))
        & (impact_stats["total"] >= 10)
    ].sort_values("total", ascending=False).head(5)
)

### Impact Distribution

Box + strip chart showing the spread of point impacts per code.

In [ ]:
ax = da.visualise_impact_boxplots(df, n_codes=15)

### High-Variance Codes

Codes with normalised std > 0.07 and at least 10 occurrences. Some are often used for both bonuses and penalties, others are likely more subjectively graded.

In [ ]:
subjective_codes = impact_stats[
    (impact_stats["count"] >= 10) & (impact_stats["std"] > 0.07)
]["code"].tolist()
print(f"High-variance codes: {subjective_codes}")
ax = da.visualise_impact_boxplots(df, codes=subjective_codes)

### Warning vs Penalty Rate

How often graders apply a code as a warning without a specified impact vs an actual deduction vs a bonus.

In [ ]:
warnings_df = da.analyse_zero_impact_warnings(df, n_codes=20)
ax = da.visualise_zero_impact_warnings(warnings_df)
da.summarise_zero_impact_warnings(warnings_df)

BLOK and SAZBA strangely contain some bonuses. Parser can be at fault in cases like this due to cases where a bonus cannot be reliably attributed to other codes. 

see raw_text column

In [ ]:
rows = df[df["code"].str.upper().isin(["BLOK", "SAZBA"]) & (df["impact"] > 0)].copy()
rows = rows.sort_values(["code", "year"])
display(
    rows[
        [
            "id",
            "year",
            "task_variant",
            "code",
            "impact",
            "impact_source",
            "raw_text",
            "comment",
        ]
    ]
    .reset_index(drop=True)
)

### Shared vs Individual Penalties

Some penalties cover a group of codes at once. This shows which codes are most shifted by shared penalties. (e.g. `STRUCT/SHORT -10` vs `STRUCT -5`)

These shared penalties are omitted from analysis elsewhere.

In [ ]:
ax = da.visualise_shared_vs_individual_impact(df)

## 5. Trends Over Time

### Code Usage Over Time

Fraction of projects that received each code per year. Shaded area shows 95% confidence interval.

Documentation-specific codes:

In [ ]:
doc_df = df[df["code"].isin(dataset_parser.DOC_CODES)]
top_doc_codes = doc_df["code"].value_counts().head(12).index.tolist()
g = da.visualise_code_usage_trends(proj, codes=top_doc_codes)

Codes in general:

In [ ]:
other_codes = (
    df[~df["code"].isin(top_doc_codes)]["code"].value_counts().head(12).index.tolist()
)
g = da.visualise_code_usage_trends(proj, codes=other_codes)

### Code Impact Over Time

Mean normalised impact per code per year, shows whether grading has become stricter or more lenient.

In [ ]:
g = da.visualise_code_impact_trends(df, n_codes=12)

## 6. Code Relationships

### Co-occurrence

Pearson correlation between code presence across projects. Codes that cluster together tend to be applied to the same submissions.

In [ ]:
ax = da.visualise_code_cooccurrence(proj, n_codes=15)

### Correlation with Documentation Score

Pearson correlation between receiving a code and the final normalised documentation score. Negative = lower score.

In [ ]:
ax = da.visualise_code_points_correlation(proj, n_codes=30)

Possible survivor bias in KAPTXT and SPACETAB.

## 7. Grader Comments

Graders can attach a free-text comment to any code.

### Comment Presence

Which codes are most frequently accompanied by an explanatory comment?

In [ ]:
comment_presence = da.analyse_comment_presence(df)

print("Codes where graders frequently add a comment:")
display(comment_presence.head(15))

print("\nCodes where graders rarely add a comment:")
display(
    comment_presence[
        (comment_presence["total_events"] >= 50)
        & (comment_presence["pct_commented"] <= 0.20)
    ]
)

### Comment Length

Distribution of comment character lengths across all code usage.

In [ ]:
comment_length_df = da.analyse_comment_length(df)
ax = da.visualise_comment_length(comment_length_df)
da.summarise_comment_length(comment_length_df).head(10)

### Language

Most comments are in Czech, with a smaller Slovak and English fraction. Done with `langdetect` library, so not too reliable. Comments that are too short, non-alphabetical or otherwise hard to pinpoint the language of are classified as unknown. 

In [ ]:
language_df = da.analyse_language_distribution(df)

In [ ]:
ax = da.visualise_language_distribution_overall(language_df)
da.summarise_language_distribution(language_df)

In [ ]:
ax = da.visualise_language_distribution_by_year(language_df)

In [ ]:
for lang_tag in ["unknown", "en"]:
    sample = (
        language_df[language_df["language"] == lang_tag]
        [["year", "task_variant", "code", "comment"]]
        .drop_duplicates(subset=["comment"])
        .head(10)
        .reset_index(drop=True)
    )
    print(f"Examples of '{lang_tag}'")
    display(sample)

### Top Keywords

Most frequent words in grader comments after Czech/Slovak stopword removal, with the codes they appear alongside most often.

In [ ]:
keywords = da.analyse_comment_keywords(df, n_keywords=30)
display(keywords)

## HOV

`HOV` (hovorový styl) is assigned when a student document contains informal, colloquial, or inappropriate language. Here are all the grader comments clarifying the code.

In [ ]:
hov_df = df.query("code == 'HOV' and not impact_shared")

hov_df = (
    hov_df.loc[hov_df["raw_text"].notna(), ["year", "task_variant", "raw_text"]]
    .drop_duplicates(subset=["raw_text"])
    .sort_values("year")
    .reset_index(drop=True)
)

# with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
#     display(hov_df)

display(hov_df)